# Title: Implementation of Dynamic Programming

## Objectives
- To understand the concept of Dynamic Programming (DP) and its two paradigms: top-down (memoization) and bottom-up (tabulation).
- To implement the All-Pairs Shortest Path algorithm (Floyd-Warshall).
- To implement the Travelling Salesman Problem (TSP) using DP with bitmask.
- To implement String Editing (Edit Distance / Levenshtein Distance).
- To implement the 0/1 Knapsack problem using Dynamic Programming.
- To implement Matrix Chain Multiplication using Dynamic Programming.
- To implement Flow Shop Scheduling (Johnson's Algorithm).
- To analyze the time and space complexity of each algorithm.

## Theory

**Dynamic Programming (DP)** is a powerful algorithmic technique for solving complex optimization problems by breaking them into overlapping subproblems and storing their solutions to avoid redundant computation. It is applicable when the problem exhibits two key properties:

1. **Optimal Substructure:** The optimal solution to the problem can be constructed from optimal solutions of its subproblems.
2. **Overlapping Subproblems:** The same subproblems are solved multiple times during recursion.

DP can be implemented in two ways:
- **Top-Down (Memoization):** Recursive approach with caching of results.
- **Bottom-Up (Tabulation):** Iterative approach that fills a table from base cases up.

**All-Pairs Shortest Path (Floyd-Warshall):** Finds the shortest path between every pair of vertices in a weighted directed graph. It iteratively relaxes paths through each vertex as an intermediate node. Time complexity: O(V³).

**Travelling Salesman Problem (TSP):** Given n cities and distances between them, find the minimum-cost tour that visits every city exactly once and returns to the start. The DP with bitmask approach stores the minimum cost to visit a subset of cities ending at a particular city. Time complexity: O(n² × 2ⁿ).

**Edit Distance (Levenshtein Distance):** Measures the minimum number of single-character edits (insertions, deletions, or substitutions) required to transform one string into another. Time complexity: O(m × n).

**0/1 Knapsack:** Given items with weights and values and a maximum capacity W, select a subset of items to maximize total value without exceeding W. Each item is either included (1) or excluded (0). Time complexity: O(n × W).

**Matrix Chain Multiplication:** Given a sequence of matrices, determine the optimal parenthesization that minimizes the total number of scalar multiplications needed to compute the product. Time complexity: O(n³).

**Flow Shop Scheduling (Johnson's Algorithm):** Schedules n jobs on 2 machines in a sequence that minimizes the total completion time (makespan). Johnson's rule orders jobs optimally by comparing processing times on both machines. Time complexity: O(n log n).

## 1. WAP to implement All-Pairs Shortest Path (Floyd-Warshall Algorithm)

### Algorithm
1. **Start**
2. Input the adjacency matrix `dist[][]` of size V×V. Set `dist[i][j] = INF` if no edge exists, and `dist[i][i] = 0`.
3. For each intermediate vertex `k` from 0 to V-1:
   - For each source vertex `i` from 0 to V-1:
     - For each destination vertex `j` from 0 to V-1:
       - If `dist[i][k] + dist[k][j] < dist[i][j]`, update `dist[i][j] = dist[i][k] + dist[k][j]`.
4. After all iterations, `dist[i][j]` holds the shortest path distance from vertex i to vertex j.
5. Display the final shortest distance matrix.
6. **Stop**

In [1]:
INF = float('inf')

def floyd_warshall(graph):
    V = len(graph)
    dist = [row[:] for row in graph]
    for k in range(V):
        for i in range(V):
            for j in range(V):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

def print_matrix(mat, label):
    V = len(mat)
    print(f'\n{label}:')
    print('      ' + '   '.join(f'V{j}' for j in range(V)))
    print('    ' + '----' * V)
    for i, row in enumerate(mat):
        vals = ['INF' if v == INF else f'{int(v):3}' for v in row]
        print(f'V{i}  | ' + '   '.join(vals))

# Graph as adjacency matrix (INF means no direct edge)
graph = [
    [0,   3,   INF, 7  ],
    [8,   0,   2,   INF],
    [5,   INF, 0,   1  ],
    [2,   INF, INF, 0  ]
]
V = len(graph)

print('All-Pairs Shortest Path - Floyd-Warshall Algorithm')
print('===================================================')
print_matrix(graph, 'Input Distance Matrix')

result = floyd_warshall(graph)
print_matrix(result, 'Shortest Path Distance Matrix (After Floyd-Warshall)')

print('\nShortest Paths Summary:')
print(f"  {'From':<6} {'To':<6} {'Distance'}")
print(f"  {'----':<6} {'--':<6} {'--------'}")
for i in range(V):
    for j in range(V):
        if i != j:
            d = result[i][j]
            print(f'  V{i:<5} V{j:<5} {"INF" if d == INF else int(d)}')


All-Pairs Shortest Path - Floyd-Warshall Algorithm

Input Distance Matrix:
      V0   V1   V2   V3
    ----------------
V0  |   0     3   INF     7
V1  |   8     0     2   INF
V2  |   5   INF     0     1
V3  |   2   INF   INF     0

Shortest Path Distance Matrix (After Floyd-Warshall):
      V0   V1   V2   V3
    ----------------
V0  |   0     3     5     6
V1  |   5     0     2     3
V2  |   3     6     0     1
V3  |   2     5     7     0

Shortest Paths Summary:
  From   To     Distance
  ----   --     --------
  V0     V1     3
  V0     V2     5
  V0     V3     6
  V1     V0     5
  V1     V2     2
  V1     V3     3
  V2     V0     3
  V2     V1     6
  V2     V3     1
  V3     V0     2
  V3     V1     5
  V3     V2     7


## 2. WAP to implement Travelling Salesman Problem (DP with Bitmask)

### Algorithm
1. **Start**
2. Input the distance matrix `dist[][]` for n cities.
3. Initialize DP table `dp[mask][i]` = minimum cost to visit all cities in `mask`, ending at city `i`. Set `dp[1][0] = 0`.
4. For each bitmask `mask` from 1 to (2^n - 1):
   - For each current city `u` in `mask`:
     - For each next city `v` not in `mask`:
       - Update `dp[mask | (1 << v)][v] = min(dp[mask|(1<<v)][v], dp[mask][u] + dist[u][v])`.
5. The answer is `min(dp[(1<<n)-1][i] + dist[i][0])` for all i.
6. Backtrack to find the actual optimal tour.
7. Display the minimum tour cost and the route.
8. **Stop**

In [2]:
INF = float('inf')

def tsp_dp(dist):
    n = len(dist)
    size = 1 << n
    dp = [[INF] * n for _ in range(size)]
    parent = [[-1] * n for _ in range(size)]
    dp[1][0] = 0

    for mask in range(1, size):
        for u in range(n):
            if not (mask & (1 << u)):
                continue
            if dp[mask][u] == INF:
                continue
            for v in range(n):
                if mask & (1 << v):
                    continue
                new_mask = mask | (1 << v)
                new_cost = dp[mask][u] + dist[u][v]
                if new_cost < dp[new_mask][v]:
                    dp[new_mask][v] = new_cost
                    parent[new_mask][v] = u

    full_mask = size - 1
    min_cost = INF
    last = -1
    for u in range(1, n):
        cost = dp[full_mask][u] + dist[u][0]
        if cost < min_cost:
            min_cost = cost
            last = u

    path = []
    mask = full_mask
    cur = last
    while cur != -1:
        path.append(cur)
        prev = parent[mask][cur]
        mask ^= (1 << cur)
        cur = prev
    path.reverse()
    path.append(0)
    return min_cost, path

dist = [
    [0,  10, 15, 20],
    [10,  0, 35, 25],
    [15, 35,  0, 30],
    [20, 25, 30,  0]
]
n = len(dist)

print('Travelling Salesman Problem - Dynamic Programming (Bitmask)')
print('============================================================')
print('Distance Matrix:')
print('     ' + '   '.join(f'C{j}' for j in range(n)))
for i, row in enumerate(dist):
    print(f'C{i}   ' + '  '.join(f'{v:3}' for v in row))

min_cost, tour = tsp_dp(dist)

print('\nOptimal Tour: ' + ' -> '.join(f'C{c}' for c in tour))
print(f'Minimum Tour Cost: {min_cost}')
print('\nTour Breakdown:')
for i in range(len(tour) - 1):
    u, v = tour[i], tour[i+1]
    print(f'  C{u} -> C{v}: distance = {dist[u][v]}')


Travelling Salesman Problem - Dynamic Programming (Bitmask)
Distance Matrix:
     C0   C1   C2   C3
C0     0   10   15   20
C1    10    0   35   25
C2    15   35    0   30
C3    20   25   30    0

Optimal Tour: C0 -> C2 -> C3 -> C1 -> C0
Minimum Tour Cost: 80

Tour Breakdown:
  C0 -> C2: distance = 15
  C2 -> C3: distance = 30
  C3 -> C1: distance = 25
  C1 -> C0: distance = 10


## 3. WAP to implement String Editing (Edit Distance / Levenshtein Distance)

### Algorithm
1. **Start**
2. Input two strings `s1` (length m) and `s2` (length n).
3. Create a DP table `dp[m+1][n+1]` where `dp[i][j]` = minimum edits to convert `s1[0..i-1]` to `s2[0..j-1]`.
4. Initialize base cases: `dp[i][0] = i` (delete all chars), `dp[0][j] = j` (insert all chars).
5. Fill the table:
   - If `s1[i-1] == s2[j-1]`: `dp[i][j] = dp[i-1][j-1]` (no operation needed).
   - Else: `dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])` (delete, insert, or substitute).
6. `dp[m][n]` gives the minimum edit distance.
7. Backtrack to find the sequence of operations performed.
8. Display the edit distance and the operation sequence.
9. **Stop**

In [3]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp

def get_operations(dp, s1, s2):
    ops = []
    i, j = len(s1), len(s2)
    while i > 0 or j > 0:
        if i > 0 and j > 0 and s1[i-1] == s2[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(f"  Substitute '{s1[i-1]}' -> '{s2[j-1]}' at position {i}")
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(f"  Delete '{s1[i-1]}' at position {i}")
            i -= 1
        else:
            ops.append(f"  Insert '{s2[j-1]}' at position {j}")
            j -= 1
    return ops[::-1]

s1 = 'SUNDAY'
s2 = 'SATURDAY'

print('String Editing - Edit Distance (Levenshtein Distance)')
print('======================================================')
print(f'Source String : "{s1}"')
print(f'Target String : "{s2}"')

dp = edit_distance(s1, s2)

print('\nDP Table:')
header = '      ' + '  '.join([' '] + list(s2))
print(header)
for i, row in enumerate(dp):
    label = s1[i-1] if i > 0 else ' '
    print(f'  {label}   ' + '  '.join(f'{v:2}' for v in row))

ops = get_operations(dp, s1, s2)
dist_val = dp[len(s1)][len(s2)]

print(f'\nMinimum Edit Distance: {dist_val}')
print(f'\nOperations Performed ({len(ops)} total):')
for op in ops:
    print(op)


String Editing - Edit Distance (Levenshtein Distance)
Source String : "SUNDAY"
Target String : "SATURDAY"

DP Table:
         S  A  T  U  R  D  A  Y
       0   1   2   3   4   5   6   7   8
  S    1   0   1   2   3   4   5   6   7
  U    2   1   1   2   2   3   4   5   6
  N    3   2   2   2   3   3   4   5   6
  D    4   3   3   3   3   4   3   4   5
  A    5   4   3   4   4   4   4   3   4
  Y    6   5   4   4   5   5   5   4   3

Minimum Edit Distance: 3

Operations Performed (3 total):
  Insert 'A' at position 2
  Insert 'T' at position 3
  Substitute 'N' -> 'R' at position 3


## 4. WAP to implement 0/1 Knapsack Problem using Dynamic Programming

### Algorithm
1. **Start**
2. Input n items with weights `w[]` and values `v[]`, and knapsack capacity `W`.
3. Create a DP table `dp[n+1][W+1]` initialized to 0.
4. For each item `i` from 1 to n:
   - For each capacity `c` from 0 to W:
     - If `w[i-1] <= c`: `dp[i][c] = max(dp[i-1][c], v[i-1] + dp[i-1][c - w[i-1]])`.
     - Else: `dp[i][c] = dp[i-1][c]`.
5. `dp[n][W]` is the maximum value.
6. Backtrack from `dp[n][W]` to find which items were selected.
7. Display the maximum value and selected items.
8. **Stop**

In [4]:
def knapsack_01(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(capacity + 1):
            if weights[i-1] <= c:
                dp[i][c] = max(dp[i-1][c], values[i-1] + dp[i-1][c - weights[i-1]])
            else:
                dp[i][c] = dp[i-1][c]
    selected = []
    c = capacity
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i-1][c]:
            selected.append(i - 1)
            c -= weights[i - 1]
    selected.reverse()
    return dp, dp[n][capacity], selected

item_names = ['Item A', 'Item B', 'Item C', 'Item D', 'Item E']
weights    = [2,        3,        4,        5,        9       ]
values     = [3,        4,        5,        6,        10      ]
capacity   = 8

print('0/1 Knapsack Problem - Dynamic Programming')
print('===========================================')
print(f"{'Item':<10} {'Weight':<10} {'Value'}")
print(f"{'----':<10} {'------':<10} {'-----'}")
for name, w, v in zip(item_names, weights, values):
    print(f'{name:<10} {w:<10} {v}')
print(f'\nKnapsack Capacity: {capacity}')

dp, max_val, selected = knapsack_01(weights, values, capacity)

print('\nDP Table (rows=items, cols=capacity 0..W):')
print(f"{'':>8}" + ''.join(f' W={c:<3}' for c in range(capacity + 1)))
print(f"{'Base':>8}" + ''.join(f'  {dp[0][c]:<4}' for c in range(capacity + 1)))
for i in range(1, len(weights) + 1):
    print(f'{item_names[i-1]:>8}' + ''.join(f'  {dp[i][c]:<4}' for c in range(capacity + 1)))

print(f'\nMaximum Value: {max_val}')
print('Selected Items:')
total_w = total_v = 0
for idx in selected:
    print(f'  {item_names[idx]}: weight={weights[idx]}, value={values[idx]}')
    total_w += weights[idx]
    total_v += values[idx]
print(f'Total Weight Used: {total_w} / {capacity}')
print(f'Total Value      : {total_v}')


0/1 Knapsack Problem - Dynamic Programming
Item       Weight     Value
----       ------     -----
Item A     2          3
Item B     3          4
Item C     4          5
Item D     5          6
Item E     9          10

Knapsack Capacity: 8

DP Table (rows=items, cols=capacity 0..W):
         W=0   W=1   W=2   W=3   W=4   W=5   W=6   W=7   W=8  
    Base  0     0     0     0     0     0     0     0     0   
  Item A  0     0     3     3     3     3     3     3     3   
  Item B  0     0     3     4     4     7     7     7     7   
  Item C  0     0     3     4     5     7     8     9     9   
  Item D  0     0     3     4     5     7     8     9     10  
  Item E  0     0     3     4     5     7     8     9     10  

Maximum Value: 10
Selected Items:
  Item B: weight=3, value=4
  Item D: weight=5, value=6
Total Weight Used: 8 / 8
Total Value      : 10


## 5. WAP to implement Matrix Chain Multiplication

### Algorithm
1. **Start**
2. Input dimensions array `p[]` of size n+1 where matrix Mi has dimensions p[i-1] × p[i].
3. Initialize DP table `m[n][n]` where `m[i][j]` = minimum multiplications for matrices i..j.
4. For each chain length `l` from 2 to n:
   - For each starting index `i` from 1 to n-l+1:
     - Set `j = i + l - 1` and `m[i][j] = INF`.
     - For each split point `k` from i to j-1:
       - Cost = `m[i][k] + m[k+1][j] + p[i-1]*p[k]*p[j]`.
       - If cost < `m[i][j]`, update `m[i][j]` and record `s[i][j] = k`.
5. `m[1][n]` gives the minimum number of scalar multiplications.
6. Use the split table to print the optimal parenthesization.
7. **Stop**

In [5]:
INF = float('inf')

def matrix_chain(p):
    n = len(p) - 1
    m = [[0] * (n + 1) for _ in range(n + 1)]
    s = [[0] * (n + 1) for _ in range(n + 1)]
    for l in range(2, n + 1):
        for i in range(1, n - l + 2):
            j = i + l - 1
            m[i][j] = INF
            for k in range(i, j):
                cost = m[i][k] + m[k+1][j] + p[i-1] * p[k] * p[j]
                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k
    return m, s

def print_optimal(s, i, j, names):
    if i == j:
        return names[i - 1]
    return '(' + print_optimal(s, i, s[i][j], names) + ' x ' + print_optimal(s, s[i][j]+1, j, names) + ')'

# Matrix dimensions: M1(30x35), M2(35x15), M3(15x5), M4(5x10), M5(10x20)
p = [30, 35, 15, 5, 10, 20]
n = len(p) - 1
matrix_names = [f'M{i+1}' for i in range(n)]

print('Matrix Chain Multiplication - Dynamic Programming')
print('=================================================')
print('Matrices and their Dimensions:')
for i in range(n):
    print(f'  {matrix_names[i]}: {p[i]} x {p[i+1]}')

m, s = matrix_chain(p)

print('\nDP Cost Table m[i][j] (minimum multiplications):')
print('      ' + '  '.join(f'j={j:2}' for j in range(1, n + 1)))
for i in range(1, n + 1):
    row = f'i={i}  '
    for j in range(1, n + 1):
        if j < i:
            row += '      '
        elif i == j:
            row += f'  {0:<4}'
        else:
            row += f'  {m[i][j]:<4}'
    print(row)

print('\nSplit Table s[i][j] (optimal split point k):')
print('      ' + '  '.join(f'j={j:2}' for j in range(1, n + 1)))
for i in range(1, n + 1):
    row = f'i={i}  '
    for j in range(1, n + 1):
        if j <= i:
            row += '      '
        else:
            row += f'  {s[i][j]:<4}'
    print(row)

print(f'\nMinimum Scalar Multiplications: {m[1][n]}')
print(f'Optimal Parenthesization: {print_optimal(s, 1, n, matrix_names)}')


Matrix Chain Multiplication - Dynamic Programming
Matrices and their Dimensions:
  M1: 30 x 35
  M2: 35 x 15
  M3: 15 x 5
  M4: 5 x 10
  M5: 10 x 20

DP Cost Table m[i][j] (minimum multiplications):
      j= 1  j= 2  j= 3  j= 4  j= 5
i=1    0     15750  7875  9375  11875
i=2          0     2625  4375  7125
i=3                0     750   2500
i=4                      0     1000
i=5                            0   

Split Table s[i][j] (optimal split point k):
      j= 1  j= 2  j= 3  j= 4  j= 5
i=1          1     1     3     3   
i=2                2     3     3   
i=3                      3     3   
i=4                            4   
i=5                                

Minimum Scalar Multiplications: 11875
Optimal Parenthesization: ((M1 x (M2 x M3)) x (M4 x M5))


## 6. WAP to implement Flow Shop Scheduling (Johnson's Algorithm)

### Algorithm
1. **Start**
2. Input n jobs, each with processing times on Machine 1 (`t1[]`) and Machine 2 (`t2[]`).
3. Divide jobs into two sets:
   - **Set U:** Jobs where `t1[i] <= t2[i]` (faster on Machine 1).
   - **Set V:** Jobs where `t1[i] > t2[i]` (faster on Machine 2).
4. Sort Set U in ascending order of `t1[i]`.
5. Sort Set V in descending order of `t2[i]`.
6. Optimal sequence = jobs in Set U (sorted) followed by jobs in Set V (sorted).
7. Calculate the makespan (total completion time) using the sequence.
8. Display the optimal job sequence and the makespan.
9. **Stop**

In [7]:
def johnsons_algorithm(jobs):
    """
    Johnson's Algorithm for 2-machine Flow Shop Scheduling.
    jobs: list of (job_id, time_machine1, time_machine2)
    """
    set_u = [j for j in jobs if j[1] <= j[2]]  # t1 <= t2
    set_v = [j for j in jobs if j[1] >  j[2]]  # t1 > t2
    set_u.sort(key=lambda x: x[1])           # ascending by t1
    set_v.sort(key=lambda x: x[2], reverse=True)  # descending by t2
    return set_u + set_v

def calculate_makespan(sequence):
    n = len(sequence)
    m1_end = [0] * n
    m2_end = [0] * n
    m1_end[0] = sequence[0][1]
    m2_end[0] = m1_end[0] + sequence[0][2]
    for i in range(1, n):
        m1_end[i] = m1_end[i-1] + sequence[i][1]
        m2_end[i] = max(m2_end[i-1], m1_end[i]) + sequence[i][2]
    return m1_end, m2_end, m2_end[-1]

# Define jobs: (job_id, time_on_M1, time_on_M2)
jobs = [
    ('J1', 5, 2),
    ('J2', 1, 6),
    ('J3', 9, 7),
    ('J4', 3, 8),
    ('J5', 10, 4)
]

print("Flow Shop Scheduling - Johnson's Algorithm (2 Machines)")
print('========================================================')
print(f"{'Job':<8} {'Machine 1 Time':<16} {'Machine 2 Time':<16} {'Set'}")
print(f"{'---':<8} {'---------------':<16} {'---------------':<16} {'---'}")
for jid, t1, t2 in jobs:
    grp = 'U (t1<=t2)' if t1 <= t2 else 'V (t1>t2) '
    print(f'{jid:<8} {t1:<16} {t2:<16} {grp}')

optimal_seq = johnsons_algorithm(jobs)

print('\nOptimal Job Sequence: ' + ' -> '.join(j[0] for j in optimal_seq))

set_u_seq = [j for j in optimal_seq if j[1] <= j[2]]
set_v_seq = [j for j in optimal_seq if j[1] > j[2]]
print('\nSet U (sorted ascending by M1 time):', [j[0] for j in set_u_seq])
print('Set V (sorted descending by M2 time):', [j[0] for j in set_v_seq])

m1_end, m2_end, makespan = calculate_makespan(optimal_seq)

print('\nGantt-style Schedule:')
print(f"  {'Job':<6} {'M1 Start':<10} {'M1 End':<10} {'M2 Start':<12} {'M2 End'}")
print(f"  {'---':<6} {'--------':<10} {'------':<10} {'--------':<12} {'------'}")
m1_start = 0
for i, j in enumerate(optimal_seq):
    m2_start = m1_end[i]
    if i > 0:
        m2_start = max(m2_end[i-1], m1_end[i])
    print(f'  {j[0]:<6} {m1_start:<10} {m1_end[i]:<10} {m2_start:<12} {m2_end[i]}')
    m1_start = m1_end[i]

print(f'\nTotal Makespan (Completion Time): {makespan}')


Flow Shop Scheduling - Johnson's Algorithm (2 Machines)
Job      Machine 1 Time   Machine 2 Time   Set
---      ---------------  ---------------  ---
J1       5                2                V (t1>t2) 
J2       1                6                U (t1<=t2)
J3       9                7                V (t1>t2) 
J4       3                8                U (t1<=t2)
J5       10               4                V (t1>t2) 

Optimal Job Sequence: J2 -> J4 -> J3 -> J5 -> J1

Set U (sorted ascending by M1 time): ['J2', 'J4']
Set V (sorted descending by M2 time): ['J3', 'J5', 'J1']

Gantt-style Schedule:
  Job    M1 Start   M1 End     M2 Start     M2 End
  ---    --------   ------     --------     ------
  J2     0          1          1            7
  J4     1          4          7            15
  J3     4          13         15           22
  J5     13         23         23           27
  J1     23         28         28           30

Total Makespan (Completion Time): 30


# Analysis of the Algorithms

| Algorithm | Time Complexity (Best) | Time Complexity (Average) | Time Complexity (Worst) | Space Complexity | Notes |
|:---|:---|:---|:---|:---|:---|
| *Floyd-Warshall (APSP)* | $O(V^3)$ | $O(V^3)$ | $O(V^3)$ | $O(V^2)$ | Handles negative edges; fails on negative cycles. |
| *TSP (Bitmask DP)* | $O(n^2 \cdot 2^n)$ | $O(n^2 \cdot 2^n)$ | $O(n^2 \cdot 2^n)$ | $O(n \cdot 2^n)$ | Exponential; exact solution; impractical for n > 20. |
| *Edit Distance* | $O(m \cdot n)$ | $O(m \cdot n)$ | $O(m \cdot n)$ | $O(m \cdot n)$ | Can reduce space to $O(\min(m,n))$ using rolling array. |
| *0/1 Knapsack* | $O(n \cdot W)$ | $O(n \cdot W)$ | $O(n \cdot W)$ | $O(n \cdot W)$ | Pseudo-polynomial; NP-Hard for large inputs. |
| *Matrix Chain Mult.* | $O(n^3)$ | $O(n^3)$ | $O(n^3)$ | $O(n^2)$ | Only finds optimal parenthesization, not the product. |
| *Flow Shop (Johnson's)* | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | Optimal for 2 machines only. |

**Notes:**
- $V$ = vertices, $n$ = number of items/cities/jobs/matrices, $m$, $n$ = string lengths, $W$ = knapsack capacity.
- All DP algorithms trade space for time by storing subproblem solutions in tables.
- TSP is NP-Hard; the bitmask DP is the fastest known exact algorithm.

# Discussion

The implemented programs demonstrated the versatility and power of Dynamic Programming across a range of classic optimization problems:

1. **Floyd-Warshall (All-Pairs Shortest Path):** The algorithm successfully computed shortest paths between all vertex pairs in a 4-vertex weighted directed graph by iteratively relaxing paths through each intermediate vertex. The O(V³) complexity makes it suitable for dense graphs where running Dijkstra's algorithm V times would be less efficient.

2. **Travelling Salesman Problem (Bitmask DP):** The bitmask DP approach encoded visited city subsets as integers, enabling efficient state representation. For 4 cities, the optimal tour was found with minimum cost by exploring all Hamiltonian paths. The exponential time complexity O(n² × 2ⁿ) limits practical use to approximately 20 cities, but it is the best known exact algorithm for TSP.

3. **Edit Distance (Levenshtein):** The DP table revealed the minimum number of insertions, deletions, and substitutions to transform "SUNDAY" into "SATURDAY". Backtracking through the table recovered the exact sequence of operations. The algorithm has wide applications in spell-checking, DNA sequence alignment, and natural language processing.

4. **0/1 Knapsack:** The bottom-up DP table systematically computed the optimal value for each item-capacity pair. The selected items and their total weight and value were recovered by backtracking. Although it runs in pseudo-polynomial time O(nW), the knapsack problem remains NP-Hard for large inputs.

5. **Matrix Chain Multiplication:** The algorithm determined the optimal split points to minimize scalar multiplications for a chain of 5 matrices, reducing computation significantly compared to naive left-to-right multiplication. The split table enables recovery of the optimal parenthesization.

6. **Flow Shop Scheduling (Johnson's Algorithm):** Johnson's algorithm partitioned jobs into two sets based on relative machine times, sorting each set to minimize idle time. The Gantt-style schedule showed start and end times on each machine. This greedy-DP hybrid is optimal for 2-machine flow shops and forms the basis for heuristic extensions to multi-machine scheduling.

# Conclusion

The experiment successfully demonstrated the implementation of six key Dynamic Programming algorithms in Python. Through Floyd-Warshall, TSP, Edit Distance, 0/1 Knapsack, Matrix Chain Multiplication, and Flow Shop Scheduling, it was observed that Dynamic Programming provides optimal solutions to complex problems by systematically decomposing them into overlapping subproblems. The trade-off between time and space complexity is central to DP design. These algorithms form the backbone of numerous real-world applications including network routing, logistics, bioinformatics, compiler optimization, and operations research. The experiment strengthened understanding of DP table construction, backtracking for solution recovery, and complexity analysis.